# Phase 4 — MongoDB

Loads the results computed in phases 3a and 3b into MongoDB, together with a
sample of the raw reviews, and queries them.

Spark writes to MongoDB directly through the official connector, so the data
never has to pass through the driver — the same approach would hold if the
collections were far larger than they are here.

Requires HDFS and `mongod` to be running (see `docs/setup.md`). The first run
downloads the connector JAR, so it needs an internet connection.

In [1]:
import getpass

from pyspark.sql import SparkSession

In [2]:
USER = getpass.getuser()
HDFS_BASE = f"hdfs://localhost:9000/user/{USER}/steam"
ANALYSIS_PATH = f"{HDFS_BASE}/output/analysis"
INPUT_PATH = f"{HDFS_BASE}/streaming_input/*.csv"

MONGO_URI = "mongodb://127.0.0.1:27017"
MONGO_DB = "steam_reviews"

spark = (
    SparkSession.builder
    .appName("steam-reviews-mongodb")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark

26/09/09 16:53:15 WARN Utils: Your hostname, luca-Katana-15-B13VFK resolves to a loopback address: 127.0.1.1; using 192.168.1.18 instead (on interface wlo1)
26/09/09 16:53:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/luca/.ivy2/cache
The jars for the packages stored in: /home/luca/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5b2eb2c9-d687-46dc-930d-14ef91fae45a;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.12;10.3.0 in central
	found org.mongodb#mongodb-driver-sync;4.8.2 in central
	[4.8.2] org.mongodb#mongodb-driver-sync;[4.8.1,4.8.99)
	found org.mongodb#bson;4.8.2 in central
	found org.mongodb#mongodb-driver-core;4.8.2 in central
	found org.mongodb#bson-record-codec;4.8.2 in central
downloading https://repo1.maven.org/maven2/org/mongodb/spark/mongo-spark-connector_2.12/10.3.0/mongo-spark-connector_2.12-10.3.0.jar ...
	[SUCCESSFUL ] org.mongodb.spark#mongo-spark-connector_2.12;10.3.0!mongo-spark-connector_2.12.jar (294ms)
downloading https://repo1.maven.org/maven2/org/mongodb/mongodb-driver-sync/4.8.2/mongodb-driver-sync-4.8.2.jar ...
	[SUCCESS

## Loading the aggregates

The six Parquet datasets written in phases 3a and 3b become six collections.
`overwrite` keeps the notebook re-runnable without piling up duplicates.

In [3]:
def write_to_mongo(df, collection):
    (
        df.write
        .format("mongodb")
        .mode("overwrite")
        .option("spark.mongodb.write.connection.uri", MONGO_URI)
        .option("database", MONGO_DB)
        .option("collection", collection)
        .save()
    )
    print(f"written: {MONGO_DB}.{collection}  ({df.count()} docs)")


AGGREGATES = [
    "recommendation_by_playtime",
    "recommendation_by_playtime_price",
    "hours_by_outcome",
    "recommendation_by_year",
    "model_metrics",
    "feature_importances",
]

for name in AGGREGATES:
    write_to_mongo(spark.read.parquet(f"{ANALYSIS_PATH}/{name}"), name)

written: steam_reviews.recommendation_by_playtime  (5 docs)
written: steam_reviews.recommendation_by_playtime_price  (20 docs)
written: steam_reviews.hours_by_outcome  (8 docs)
written: steam_reviews.recommendation_by_year  (10 docs)
written: steam_reviews.model_metrics  (2 docs)
written: steam_reviews.feature_importances  (7 docs)


## Loading a sample of the raw reviews

The aggregates alone would only allow trivial queries, so a slice of the
review-level data goes in as well.

In [4]:
from pyspark.sql import functions as F

raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(INPUT_PATH)
)

reviews_sample = (
    raw
    .select(
        "app_id", "title", "hours", "is_recommended", "helpful",
        "price_final", "price_bucket", "positive_ratio", "date",
    )
    .sample(fraction=0.1, seed=42)
)

write_to_mongo(reviews_sample, "reviews")

[Stage 35:>                                                       (0 + 16) / 17]

written: steam_reviews.reviews  (50097 docs)


## Reading back

A round-trip check that the connector wrote what we expect.

In [5]:
check = (
    spark.read
    .format("mongodb")
    .option("spark.mongodb.read.connection.uri", MONGO_URI)
    .option("database", MONGO_DB)
    .option("collection", "recommendation_by_playtime")
    .load()
)

check.show()

+--------------------+---------------+------+-----------+------+
|                 _id|playtime_bucket|  rate|recommended| total|
+--------------------+---------------+------+-----------+------+
|6aa172f6a05b8f573...|           0-1h| 0.351|       2326|  6626|
|6aa172f6a05b8f573...|        20-100h|0.8661|     137418|158660|
|6aa172f6a05b8f573...|           1-5h|0.5748|      10625| 18484|
|6aa172f6a05b8f573...|          100h+|0.8711|     218341|250650|
|6aa172f6a05b8f573...|          5-20h|0.8221|      53913| 65580|
+--------------------+---------------+------+-----------+------+

